In [1]:
import pandas as pd
from pathlib import Path
import util

pd.set_option('display.float_format', '{:,.0f}'.format)

In [2]:
# network data
df_network = util.process_network_summary()

In [3]:
county_order = ['King','Kitsap','Pierce','Snohomish','Outside Region','Total']

df_network['county'] = pd.Categorical(df_network['county'], ordered=True,
                   categories=county_order)

In [4]:
#| label: system_summary

# network summary: 'VMT','VHT','total_delay'
vmt = df_network['VMT'].sum()
vht = df_network['VHT'].sum()
delay = df_network['total_delay'].sum()

# Get light rail boardings
df_boardings = pd.read_csv(util.output_path / 'transit/daily_boardings_by_agency.csv')
# transit boardings
transit_ridership = df_boardings['boardings'].sum()
#Light rail boardings
df_line_boardings = pd.read_csv(util.output_path / 'transit/transit_line_results.csv')
lr_ridership = df_line_boardings[df_line_boardings['mode']=='r']['boardings'].sum()

# mode share
df_trip = pd.read_csv(util.output_path / 'agg/dash/mode_share_county.csv')
df = df_trip['trexpfac'].sum() # total trips
transit_share = df_trip.loc[df_trip['mode']=="Transit"]['trexpfac'].sum()/df

# emission
df_emissions = pd.read_csv(util.output_path / 'emissions/emissions_summary.csv')
df = df_emissions.loc[(df_emissions['veh_type'].isin(['light','medium','heavy'])) & \
                      (df_emissions['pollutant_name']=="CO2 Equivalent")].copy()
CO2e = df['total_daily_tons'].sum()

run_summary2 = pd.DataFrame({
    'VMT': [vmt],
    'VHT': [vht],
    'Delay': [delay],
    'Transit Boardings': [transit_ridership],
    'Light Rail Boardings': [lr_ridership],
    '% Transit': [transit_share],
    'CO2e': [CO2e]
})

run_summary2.style.\
        format('{:,.0f}', subset=['VMT','VHT','Delay','Transit Boardings','Light Rail Boardings','CO2e']).\
        format('{:.1%}', subset=['% Transit']).\
            hide(axis="index")

VMT,VHT,Delay,Transit Boardings,Light Rail Boardings,% Transit,CO2e
"83,161,952","2,548,050","224,337","514,012","88,955",2.2%,"39,501"


In [5]:
county_order = ['King','Kitsap','Pierce','Snohomish','Outside Region','Total']

df_network['county'] = pd.Categorical(df_network['county'], ordered=True,
                   categories=county_order)

In [6]:
#| label: total_vmt_vht_delay_by_county

df_vmt = df_network.reset_index().groupby('county',observed=True)['VMT'].sum()
df_vht = df_network.reset_index().groupby('county',observed=True)['VHT'].sum()
df_delay = df_network.reset_index().groupby('county',observed=True)['total_delay'].sum()

df = pd.concat([df_vmt,df_vht,df_delay], axis=1)
df = df[df.index!='Outside Region']
df.rename(columns={'total_delay': 'Total Delay Hours'}, inplace=True)

df.loc['Total',] = df.sum()
total_vmt = df.loc['Total','VMT']

df.style.format('{:,.0f}')

,VMT,VHT,Total Delay Hours
county,,,
King,"43,893,946","1,394,458","159,386"
Kitsap,"4,347,451","128,398","2,850"
Pierce,"18,348,951","546,855","30,926"
Snohomish,"16,234,937","471,826","31,175"
Total,"82,825,285","2,541,537","224,337"


In [7]:
#| label: transit
transit = pd.read_csv(util.output_path / 'transit/daily_boardings_by_agency.csv')

df = transit[['agency_name','boardings']].set_index('agency_name').copy()
df.loc['Total',:] = df['boardings'].sum()
df.rename(columns={'boardings': 'Daily Boardings'}, inplace=True)


df.style.format('{:,.0f}')


,Daily Boardings
agency_name,
King County Metro,"291,482"
Sound Transit,"142,454"
Community Transit,"26,799"
Pierce Transit,"24,732"
Kitsap Transit,"14,101"
Washington Ferries,"9,494"
Everett Transit,"4,951"
Total,"514,012"


In [8]:
#| label: mode_share
mode_share = pd.read_csv(util.output_path / 'agg/dash/mode_share_county.csv')

df = mode_share.groupby('mode')['trexpfac'].sum().reset_index().set_index('mode')
df['Mode Share'] = (df['trexpfac']/ (mode_share['trexpfac'].sum()))
df.loc['Total', :] = df.sum(axis=0)
df['Mode Share'] = (df['Mode Share'] * 100).round(1)
df['Mode Share'] = df['Mode Share'].astype(str) + '%'
df['trexpfac'] = df['trexpfac'].apply(lambda x: f"{int(round(x)):,}")
df.rename(columns={'trexpfac': 'Total Person Trips'}, inplace=True)
df

,Total Person Trips,Mode Share
mode,,
Bike,"246,114",1.5%
HOV2,"3,469,043",21.2%
HOV3+,"2,296,680",14.1%
SOV,"7,354,962",45.0%
School Bus,"258,066",1.6%
Transit,"357,927",2.2%
Walk,"2,358,601",14.4%
Total,"16,341,393",100.0%
